In [9]:
from langchain.llms import GooglePalm
import os

In [17]:
import google.generativeai as genai
import os

# Assuming the API key is already set in the environment from a previous cell
# If not, you would need to set it here, potentially using getpass or userdata.get
api_key = os.environ.get("GOOGLE_API_KEY")
if not api_key:
  # You might want to handle this case more robustly, e.g., raise an error or prompt the user
  print("Google API key not found in environment variables.")
else:
  genai.configure(api_key=api_key)
  # Using google.generativeai directly to generate text
  model = genai.GenerativeModel('gemini-1.5-flash-latest') # Using a valid and supported model name
  response = model.generate_content("Write a 4 line poem of my love for samosa")
  print(response.text)

Golden brown, a crispy shell,
Spiced potato, magic spell.
Savory delight, a perfect bite,
My love for samosa, shining bright.



In [18]:
from langchain.chains import RetrievalQA


from langchain.embeddings import GooglePalmEmbeddings
from langchain.llms import GooglePalm

In [19]:
from langchain.document_loaders.csv_loader import CSVLoader

# Mount Google Drive if not already done
from google.colab import drive
drive.mount('/content/drive')

# Correct the file path (Colab uses '/content/drive/My Drive/')
file_path = '/content/drive/My Drive/rag/data/4c_research_lab_faqs.csv'

# Load CSV file
loader = CSVLoader(file_path=file_path)

# Store the loaded data in the 'data' variable
data = loader.load()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize instructor embeddings using the Hugging Face model
instructor_embeddings = HuggingFaceEmbeddings(model_name="hkunlp/instructor-large")

e = instructor_embeddings.embed_query("What is your refund policy?")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.15M [00:00<?, ?B/s]

In [24]:
len(e)

768

In [25]:
e[:5]

[-0.04449572414159775,
 0.0076915244571864605,
 -0.009869124740362167,
 0.020831886678934097,
 0.03185894712805748]

In [26]:
from langchain.vectorstores import FAISS

# Create a FAISS instance for vector database from 'data'
vectordb = FAISS.from_documents(documents=data,
                                 embedding=instructor_embeddings)

# Create a retriever for querying the vector database
retriever = vectordb.as_retriever(score_threshold = 0.7)

In [27]:
rdocs = retriever.get_relevant_documents("how about job placement support?")
rdocs

/tmp/ipython-input-27-3956196717.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  rdocs = retriever.get_relevant_documents("how about job placement support?")


[Document(id='defd0016-534b-48f0-8eb4-40aa053f59b4', metadata={'source': '/content/drive/My Drive/rag/data/4c_research_lab_faqs.csv', 'row': 44}, page_content='question: What types of research positions are available?\nanswer: Available positions include: 1) Research assistants, 2) Graduate students, 3) Postdoctoral fellows, 4) Clinical research coordinators, 5) Data analysts, and 6) Volunteer research positions.'),
 Document(id='45d99cc1-550c-4556-ae9d-bcf09b8abcec', metadata={'source': '/content/drive/My Drive/rag/data/4c_research_lab_faqs.csv', 'row': 56}, page_content="question: What is the lab's approach to mentorship?\nanswer: Our mentorship approach includes: 1) Individual guidance, 2) Skill development, 3) Career planning, 4) Research training, 5) Professional networking, and 6) Continuous support."),
 Document(id='fe49d0c6-019c-42ba-9ef8-e85a8ed07025', metadata={'source': '/content/drive/My Drive/rag/data/4c_research_lab_faqs.csv', 'row': 43}, page_content='question: How can I

In [28]:
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain.llms import GooglePalm
import os

# Assuming the API key is already set in the environment from a previous cell
# If not, you would need to set it here, potentially using getpass or userdata.get
api_key = os.environ.get("GOOGLE_API_KEY")
if not api_key:
  # You might want to handle this case more robustly, e.g., raise an error or prompt the user
  print("Google API key not found in environment variables.")
else:
  llm = GooglePalm(google_api_key=api_key, temperature=0.7)


prompt_template = """Given the following context and a question, generate an answer based on this context only.
In the answer try to provide as much text as possible from "response" section in the source document context without making much changes.
If the answer is not found in the context, kindly state "I don't know." Don't try to make up an answer.

CONTEXT: {context}

QUESTION: {question}"""


PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)
chain_type_kwargs = {"prompt": PROMPT}


chain = RetrievalQA.from_chain_type(llm=llm,
                            chain_type="stuff",
                            retriever=retriever,
                            input_key="query",
                            return_source_documents=True,
                            chain_type_kwargs=chain_type_kwargs)

In [ ]:
# Use the functions defined in previous cells to perform RAG
query = 'Do you provide job assistance ?'
context = retrieve_context(query)
answer = generate_answer_with_gemini(context, query)
print(answer)

I don't know.



In [40]:
print(ask_rag("How to contact 4C lab?"))

To discuss potential collaborations, research partnerships, or clinical trials, contact Dr. Rishi Ganesan at rishi.ganesan@lhsc.on.ca.  To join the research team, send your CV to the same email address.



In [42]:
print(ask_rag("Can I join the lap as a Phd student?"))

I don't know.



In [8]:
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.4 MB/s eta 0:00:00


In [21]:
%pip install --quiet InstructorEmbedding langchain-huggingface

In [22]:
%pip install --quiet faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 62.1 MB/s eta 0:00:00
